# 11 — Agente central e três domínios AWS

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 4 contas AWS distintas

**Esta aula não executa em uma conta única.** O fluxo mostrado envolve 4 contas (platform, retail-banking, transaction-banking, lending-wealth), mas o manifesto e o IaC atuais exigem as quatro contas preparadas: plataforma, Retail, Transaction e Lending. Sem elas, reserve a aula para demonstração do instrutor no ambiente completo ou retire-a da seleção de execução.

**Runner CodeBuild do projeto:** a implantação atual exige quatro contas e as integrações Registry, Terraform e RAG, inclusive com `--labs`. A seleção parcial não converte esse runner em uma implantação de conta única. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 11](../docs/diagramas/svg/11_blog_multi_account_gateway_mcp.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "11")
print({"lab": "11", "dependencies": list(dependencies), "region": config["region"]})

### Conectar ao ambiente implantado

Lemos os outputs exatos do CloudFormation. O cliente não cria infraestrutura e não substitui um serviço indisponível.

In [ ]:
lab = AWSWorkshop(CONFIG, domain="platform")
print(lab.preflight())

### Autenticar a pessoa no Cognito

O login usa USER_PASSWORD_AUTH para usuários sintéticos do workshop e SECRET_HASH para o cliente confidencial. Em uma aplicação de usuários finais, use authorization code com PKCE. O código abaixo mostra a chamada Cognito e a verificação de assinatura, emissor, audiência e token_use. Só imprimimos claims selecionadas.

In [ ]:
from santander_demo.aws_workshop.identity import login, verify
display(Code(inspect.getsource(login), language="python"))
display(Code(inspect.getsource(verify), language="python"))
token = lab.login("c", "supervisor")
claims = lab.claims(token)
print({key: claims[key] for key in ["iss", "sub", "aud", "token_use"]})

### Inspecionar os targets do hub

O Gateway da plataforma agrega os Gateways de Retail, Transaction e Lending. O schema está no IaC; não precisamos de um token humano durante a criação dos targets.

In [ ]:
display(Code((ROOT / "infra/workshop/templates/federation-bindings.json").read_text(), language="json"))
names = [x["name"] for x in await lab.tools(token)]
for prefix in ["retail___", "transaction___", "lending___"]:
    assert any(name.startswith(prefix) for name in names)
print(names)

### Fazer o agente consultar três contas

O notebook envia perguntas. Escolha de ferramenta, inferência Bedrock e chamadas MCP executam no Runtime da plataforma.

In [ ]:
results = {}
for domain in ["retail-banking", "transaction-banking", "lending-wealth"]:
    result = await lab.invoke_agent(token, "Consulte o risco sintético de C001.", domain=domain)
    assert result["execution"] == "AWS_AGENTCORE_RUNTIME"
    results[domain] = result
print(json.dumps(results, indent=2, ensure_ascii=False))

**Relação com os exemplos oficiais:** o serviço Gateway centraliza a descoberta MCP. Aqui preservamos a pessoa com JWT Cognito e políticas independentes. O fluxo M2M de aplicação é demonstrado separadamente no laboratório 09; não equivale a OBO.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.